# 19 A Small Program, Built One Step at a Time

| | |
|---|---|
| **Path** | Python Fundamentals |
| **Prerequisite** | 18 |

The earlier lessons each added one tool. This lesson uses those tools in the order a program actually grows. Run the cells from top to bottom. Each cell adds one function and uses what is already defined.

The program counts words in a piece of text.

1. Normalize one word.
2. Count the words in a string.
3. Choose the most common word.
4. Read the text from a file.
5. Refuse an empty file with a clear error.

A word, for this program, is a run of letters. Digits and punctuation are dropped, and case does not matter.


## Step 1 — Normalize one word

### Predict

`"Ada,"` loses the comma and becomes `"ada"`. `"1843"` has no letters and becomes an empty string. `"NOTE"` becomes `"note"`.

### Run


In [1]:
def normalize(word):
    # Keep letters only, and compare words without case.
    letters = []
    for character in word.lower():
        if character.isalpha():
            letters.append(character)
    return "".join(letters)

print(normalize("Ada,"))
print(repr(normalize("1843")))
print(normalize("NOTE"))


ada
''
note


## Step 2 — Count words

### Predict

The sentence has `ada` twice and `grace` once. The number `1843` disappears during normalization, so it is not a key. Empty pieces are skipped.

### Run


In [2]:
def counts(text):
    totals = {}
    for raw in text.split():
        word = normalize(raw)
        # A token with no letters is not a word.
        if word == "":
            continue
        totals[word] = totals.get(word, 0) + 1
    return totals

sample = "Ada met Ada. Grace arrived in 1843."
print(counts(sample))


{'ada': 2, 'met': 1, 'grace': 1, 'arrived': 1, 'in': 1}


## Step 3 — Choose the most common word

### Predict

`ada` has the larger count, so the winner is the pair `("ada", 2)`. An empty dictionary has no winner and raises `ValueError`. If two words tie, the one that was inserted first wins, because a later equal count is not greater.

### Run


In [3]:
def most_common(totals):
    if not totals:
        raise ValueError("no words")
    winner = None
    for word, count in totals.items():
        # A later tie does not replace the winner. Only a greater count does.
        if winner is None or count > totals[winner]:
            winner = word
    return winner, totals[winner]

print(most_common(counts("Ada met Ada. Grace arrived.")))
try:
    print(most_common({}))
except ValueError as error:
    print(error)


('ada', 2)
no words


## Step 4 — Read the text from a file

### Predict

The file holds two lines. `read_text` returns them as one string, `counts` walks the words, and `most_common` still reports `ada` with count 2.

### Run


In [4]:
import tempfile
from pathlib import Path

def report(path):
    # Reading the file stays at the edge. Counting stays in counts.
    text = path.read_text(encoding="utf-8")
    return most_common(counts(text))

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "notes.txt"
    path.write_text("Ada met Ada.\nGrace arrived.\n", encoding="utf-8")
    print(report(path))


('ada', 2)


## Step 5 — An empty file is a real result, reported on purpose

### Predict

The file exists and contains no words. `counts` returns an empty dictionary. `most_common` raises, and `report_or_explain` turns that into the sentence `no words` instead of a traceback.

### Run


In [5]:
import tempfile
from pathlib import Path

def report_or_explain(path):
    try:
        text = path.read_text(encoding="utf-8")
        word, count = most_common(counts(text))
    except ValueError as error:
        # An empty count is explained instead of shown as a traceback.
        return str(error)
    return f"{word}: {count}"

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "empty.txt"
    path.write_text("... 1843 ...\n", encoding="utf-8")
    print(counts(path.read_text(encoding="utf-8")))
    print(report_or_explain(path))


{}
no words


## A pitfall

Splitting on spaces is not the same as normalizing. `"Ada,"` and `"Ada"` would be two keys if the comma stayed attached. The count of `ada` would be wrong. Normalization is what makes them one word.

### Run


In [6]:
text = "Ada, Ada"
naive = {}
# Splitting alone keeps the comma attached, so the two tokens differ.
for raw in text.split():
    naive[raw] = naive.get(raw, 0) + 1
print(naive)
print(counts(text))


{'Ada,': 1, 'Ada': 1}
{'ada': 2}


## Summary

A small program is a stack of functions, each with one job. Normalize before you count, count before you rank, and read the file at the edge of the program rather than inside the counting rule. An empty result is a `ValueError` you can explain, not a special case scattered through every loop.
